# Rapid proteomic analysis for solid tumors reveals LSD1 as a drug target in an end‐stage cancer patient
#### Doll et al. https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6068348/

### Abstract
Recent advances in mass spectrometry (MS)‐based technologies are now set to transform translational cancer proteomics from an idea to a practice. Here, we present a robust proteomic workflow for the analysis of clinically relevant human cancer tissues that allows quantitation of thousands of tumor proteins in several hours of measuring time and a total turnaround of a few days. We applied it to a chemorefractory metastatic case of the extremely rare urachal carcinoma. Quantitative comparison of lung metastases and surrounding tissue revealed several significantly upregulated proteins, among them lysine‐specific histone demethylase 1 (LSD1/KDM1A). LSD1 is an epigenetic regulator and the target of active development efforts in oncology. Thus, clinical cancer proteomics can rapidly and efficiently identify actionable therapeutic options. While currently described for a single case study, we envision that it can be applied broadly to other patients in a similar condition.

### Workflow

![Figure 2](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6068348/bin/MOL2-12-1296-g002.jpg)

**Figure 2** Proteomics workflow for the case study. (A) Timeline of the project. (B) Experimental design, including source of material, inStageTip sample preparation, and depiction of the analytical workflow


### Results

![Figure 3](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC6068348/bin/MOL2-12-1296-g003.jpg)

**Figure 3** Proteins differentially expressed in the urachal carcinoma lung metastases. (A) Volcano plot of the p‐values (y‐axis) vs. the log2 protein abundance differences (x‐axis) between metastases and control, with lines of significance colored in black or gray lines corresponding to a 5% or 1% FDR, respectively. (B) Mechanisms of action of LSD1/KDM1A and inhibitory drug treatment proposed: **JATROSOME. TRANYLCYPROMIN**


### Workflow with the Clinical Knowledge Graph

1. Generate Analysis Report: Proteomics data
2. Identify Candidate Drug Treatments
3. Rank Candidates According to Toxicity

![Clinical_Knowledge_Graph](banner.jpg)

## Generate Analysis Report: Proteomics Data

### Report Manager

In [ ]:
from report_manager import project

##### We load specific configuration for this project.

In [ ]:
configuration_files = {"proteomics":"./local-data)/code/src/report_manager/config/proteomics_P0000002.yml"}

##### We create a new project object that we can use to run the analyses

In [ ]:
study_case_project = project.Project(identifier="P0000002", configuration_files=configuration_files, datasets={}, knowledge=None, report={})

##### We need to first build the project. This step will collect the project datasets from CKG and process them

In [ ]:
study_case_project.build_project(force=False)

##### We can now generate the report following the specified configuration

In [ ]:
study_case_project.generate_report()

##### Ready! All the analyses are done and we can now access to all the results for each data type

In [ ]:
study_case_project.list_datasets()

##### We will use the results from the proteomics analyses. We access the dataset 'proteomics' for further analysis.

In [ ]:
proteomics_dataset = study_case_project.get_dataset(dataset='proteomics')

##### The available analysis for this dataset are:

In [ ]:
proteomics_dataset.list_dataframes()

###### In this case, we use the regulation table to extract proteins upregulated in the metastatic tissue compare to non-cancerous tissue.
***Note: the automated analysis from CKG return the comparison CONTROL vs CANCER***

In [ ]:
regulation_table = proteomics_dataset.get_dataframe(dataset_name='regulation table')

In [ ]:
regulation_table.head()

In [ ]:
regulation_table[regulation_table["identifier"].str.startswith('KDM1A')]

##### As in the article, we use significantly regulated proteins with a fold change higher than two.

In [ ]:
up_regulated_proteins = regulation_table.loc[(regulation_table.rejected) & (regulation_table.FC < -2), ['identifier']]

In [ ]:
up_regulated_proteins.shape

### Graph Database Connector

In [ ]:
from graphdb_connector import query_utils, connector

##### We connect to CKG database using the default configuration

In [ ]:
driver = connector.getGraphDatabaseConnectionConfiguration()

##### We load the existing database queries that we can use to extract knowledge from CKG

In [ ]:
queries = query_utils.read_knowledge_queries()

### 1) Filter for Regulated Proteins Associated to Lung Cancer:

##### We want to check whether we can identify known connection between the upregulated proteins in metastases and the disease
##### We check if there are queries for these node types: Protein, Disease

In [ ]:
selected_queries = query_utils.find_queries_involving_nodes(queries=queries, nodes=["Protein", "Disease"], print_pretty=True)

##### The query named 'specific disease' can help us in this case

In [ ]:
disease_query = selected_queries["association_disease_score"]["query"]
proteins = ['"{}"'.format(p) for p in up_regulated_proteins["identifier"].tolist()]
diseases = ['DOID:1324']
diseases = ['"{}"'.format(d) for d in diseases]
disease_query = disease_query.format(",".join(proteins),",".join(diseases), 1)

In [ ]:
proteins_associated_lung_cancer = connector.getCursorData(driver=driver, query=disease_query, parameters={})

In [ ]:
proteins_associated_lung_cancer.head()

In [ ]:
proteins_associated_lung_cancer.shape

### 2) Identify Inhibitory Drugs for those Proteins

##### We use again the functionality 'find_queries_involving_nodes' to find queries involving nodes: Protein, Drug

In [ ]:
selected_queries = query_utils.find_queries_involving_nodes(queries=queries, nodes=["Protein", "Drug"], print_pretty=True)

In [ ]:
proteins = ['"{}"'.format(p) for p in proteins_associated_lung_cancer['node1'].tolist()]
drug_query = queries["association_drug_interaction_score"]["query"].format(",".join(proteins), 'inhibition', 0.8)

##### We search in CKG database for known inhibitory drugs for these proteins

In [ ]:
drugs_proposed = connector.getCursorData(driver=driver, query=drug_query, parameters={})

In [ ]:
drugs_proposed.head()

In [ ]:
drugs_proposed.shape

##### These list of inhibitory drugs could in principle be used to identify alternative treatments
#### We can already see that CKG found the same inhibitory drug that was identified in the study case published. However, many other options are proposed and could be further ranked using other criteria.

In [ ]:
from analytics_core import utils
from analytics_core.viz import viz

In [ ]:
net = viz.get_network(data=drugs_proposed, identifier="inhibition_drugs", args={"source":"node1", "target":"node2", "values":"weight", "node_size":"degree","title":"Proposed drugs", "color_weight":False})

In [ ]:
viz.visualize_notebook_network(net["notebook"], notebook_type='jupyter', layout={'width':'100%', 'height':'700px'})

### 3) Identify Drug Known Side Effects

##### In the case study, toxicity was in part the reason why the treatment regimens did not work. We could use the list of side effects to prioritize these drugs.

##### Let's find database queries to obtain these associations: Phenotype (side effect), Drug.

In [ ]:
selected_queries = query_utils.find_queries_involving_nodes(queries=queries, nodes=["Phenotype", "Drug"], print_pretty=True)

In [ ]:
drugs = drugs_proposed["drug_id"].unique()
drugs = ['"{}"'.format(d) for d in drugs]
sideeffects_query = queries["association_drug_sideeffects"]["query"].format(",".join(drugs))

In [ ]:
side_effects = connector.getCursorData(driver=driver, query=sideeffects_query, parameters={})

In [ ]:
side_effects.head()

##### The treatment regimens are also available in CKG and their side effects can be used to rank the proposed drugs. We can prioritize drugs with side effects dissimilar to the ones that caused an adverse reaction in the patient.

##### We will in this case define a new query to obtain the treatment intervention

In [ ]:
treatment_intervention='''MATCH (project:Project)-[:HAS_ENROLLED_SUBJECT]-
                        (subject:Subject)-[:HAD_INTERVENTION]-(treatment:Clinical_variable)
                        WHERE project.id="P0000002
                        RETURN treatment.name AS treatment"'''
side_effects_similarity = '''MATCH (d1:Drug)-[:HAS_SIDE_EFFECT]->(phenotype1)
                            WHERE d1.name in [{}]
                            WITH d1, unique(collect(id(phenotype1))) AS d1sideeffects
                            MATCH (d2:Drug)-[:HAS_SIDE_EFFECTS]->(phenotype2)
                            WHERE d2.id IN [{}]
                            WITH d1, d1sideeffects, d2, collect(id(phenotype2)) AS d2sideeffects
                            RETURN d1.name AS from,
                                   d2.name AS to,
                                   algo.similarity.jaccard(d1sideeffects, d2sideeffects) AS similarity'''

In [ ]:
net = viz.get_network(data=side_effects, identifier="side_effects", args={"source":"node1", "target":"node2", "node_size":"degree","title":"Side effects", "color_weight":False})

In [ ]:
viz.visualize_notebook_network(net["notebook"], notebook_type='jupyter', layout={'width':'100%', 'height':'700px'})

In [ ]:
net_json = net['net_json']
graph_nx = utils.json_network_to_networkx(net_json)
sorted_degrees = sorted(graph_nx.degree, key=lambda x: x[1], reverse=False)
count = 30
top_20_lowest_degree = []
for n, degree in sorted_degrees:
    if n in side_effects["node1"].unique():
        top_20_lowest_degree.append((n,degree))
        count -=1
    if count==0:
        break

In [ ]:
top_20_lowest_degree

In [ ]:
set(top_20_highest_degree).intersection(top_20_lowest_degree)